In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
# adjust paths if needed
features = pd.read_parquet("../../data/processed/market_all_features_with_sentiment.parquet")
features.head()

In [ ]:
# If some entries are the string "None", turn them into real NaN first (just in case)
features["sent_cat_lag1"] = features["sent_cat_lag1"].replace("None", np.nan)

# 1) Category lag: treat missing as NEUTRAL
features["sent_cat_lag1"] = features["sent_cat_lag1"].fillna("neutral")

# 2) Numeric lag: treat missing as 0 (neutral score)
features["sent_lag1"] = features["sent_lag1"].fillna(0.0)

# 3) Rolling sentiment: also neutral when missing
features["sent_roll3"] = features["sent_roll3"].fillna(0.0)

# Quick check
print(features[["sent_daily_cat", "sent_lag1", "sent_cat_lag1", "sent_roll3"]].head())
print(features[["sent_cat_lag1"]].isna().sum())


In [ ]:
features.head()

## Create target and lag features

In [ ]:
# === 1) Ensure correct ordering ===
# (date should already be datetime, but this is safe)
features["date"] = pd.to_datetime(features["date"], errors='coerce')

# Sort by ticker and date to make all shifts deterministic
features = features.sort_values(["ticker", "date"]).reset_index(drop=True)

# We'll work on a copy for modeling features
features_model = features.copy()

# === 2) Target: next-day return per ticker (t+1) ===
features_model["target_ret_next"] = features_model.groupby("ticker")["return"].shift(-1)

# === 3) Autoregressive lags of return (AR(5) style) ===
for lag in range(1, 6):
    features_model[f"ret_lag{lag}"] = features_model.groupby("ticker")["return"].shift(lag)

# === 4) Optional: lagged volatility (21-day) ===
features_model["vol21_lag1"] = features_model.groupby("ticker")["vol21"].shift(1)

# (⚠️ Do NOT drop NaNs yet; we will drop them
#  separately in train/test to keep the split consistent.)

print(features_model.shape)
features_model[[
    "date", "ticker",
    "return", "target_ret_next",
    "ret_lag1", "ret_lag2", "ret_lag3", "ret_lag4", "ret_lag5",
    "vol21", "vol21_lag1"
]].head(10)


### Build train / test modeling DataFrames using existing split 

In [ ]:
# Sanity: check that 'set' is still present
features_model[["set"]].value_counts()

In [ ]:
# 2.1 Create train/test based on fixed split
train_df = features_model[features_model["set"] == "train"].copy()
test_df  = features_model[features_model["set"] == "test"].copy()

In [ ]:
train_df.ticker.value_counts()

In [ ]:
test_df.ticker.value_counts()

In [ ]:
print("Raw shapes (before dropping NaNs):")
print("train_df:", train_df.shape)
print("test_df :", test_df.shape)

In [ ]:
# 2.2 For AR models, we at least need:
required_cols = ["target_ret_next"] + [f"ret_lag{i}" for i in range(1, 6)]
required_cols

In [ ]:
# Drop rows with missing target or AR lags *inside each set*
train_df = train_df.dropna(subset=required_cols).copy()
test_df  = test_df.dropna(subset=required_cols).copy()

In [ ]:
print("\nAfter dropping rows with missing target/AR lags:")
print("train_df:", train_df.shape)
print("test_df :", test_df.shape)

In [ ]:
train_df.ticker.value_counts()

In [ ]:
test_df.ticker.value_counts()

In [ ]:
# Quick preview
train_df[[
    "date", "ticker",
    "return", "target_ret_next",
    "ret_lag1", "ret_lag2", "ret_lag3", "ret_lag4", "ret_lag5"
]].head()

## Baseline AR

### Baseline (market-only) feature set

In [ ]:
baseline_features = [
    # AR lags
    "ret_lag1", "ret_lag2", "ret_lag3", "ret_lag4", "ret_lag5",
    
    # volatility features
    "vol21", "vol63", "vol21_lag1",
    
    # trend features
    "ret_ema21", "ret_ema63",
    "px_ma21", "px_ma63", "px_ma200",
    "trend200_up",
]


### Prepare X_train_base, X_test_base

In [ ]:
# Drop NaN rows ONLY inside the split
train_base = train_df.dropna(subset=baseline_features + ["target_ret_next"]).copy()
test_base  = test_df.dropna(subset=baseline_features + ["target_ret_next"]).copy()

X_train_base = train_base[baseline_features]
y_train_base = train_base["target_ret_next"]

X_test_base = test_base[baseline_features]
y_test_base = test_base["target_ret_next"]

X_train_base.shape, X_test_base.shape


### Scale baseline features (train-only fit)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Columns that will be scaled
scale_cols_base = baseline_features  # all numeric in baseline

# Initialize scaler
scaler_base = StandardScaler()

# Fit scaler ONLY on the training data
scaler_base.fit(X_train_base[scale_cols_base])

# Transform both train and test
X_train_base_scaled = X_train_base.copy()
X_test_base_scaled = X_test_base.copy()

X_train_base_scaled[scale_cols_base] = scaler_base.transform(X_train_base[scale_cols_base])
X_test_base_scaled[scale_cols_base] = scaler_base.transform(X_test_base[scale_cols_base])

X_train_base_scaled.head()


### Fit Linear Regression Baseline Model

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Initialize linear regression
lin_base = LinearRegression()

# Fit on training data
lin_base.fit(X_train_base_scaled, y_train_base)

# Predict on test
y_pred_base = lin_base.predict(X_test_base_scaled)


### Evaluate Baseline Model

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def directional_accuracy(y_true, y_pred):
    return np.mean(np.sign(y_true) == np.sign(y_pred))

# ==== Compute metrics ====

# MSE and RMSE
mse_base  = mean_squared_error(y_test_base, y_pred_base)
rmse_base = np.sqrt(mse_base)

# MAE
mae_base = mean_absolute_error(y_test_base, y_pred_base)

# R²
r2_base = r2_score(y_test_base, y_pred_base)

# Directional Accuracy
dir_acc_base = directional_accuracy(y_test_base, y_pred_base)

# ==== Print results ====
print("=== Baseline Linear AR (Market-only) ===")
print(f"MSE : {mse_base:.6f}")
print(f"RMSE: {rmse_base:.6f}")
print(f"MAE : {mae_base:.6f}")
print(f"R²  : {r2_base:.4f}")
print(f"Directional accuracy: {dir_acc_base:.4f}")



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plot_df = test_base.copy()
plot_df["pred_base"] = y_pred_base
plot_df = plot_df.sort_values("date")

sns.set(style="whitegrid", context="talk")

plt.figure(figsize=(16,6))
sns.lineplot(x="date", y="target_ret_next", data=plot_df, label="Actual Next-Day Return")
sns.lineplot(x="date", y="pred_base", data=plot_df, label="Predicted Return (Baseline AR)")
plt.title("Actual vs Predicted Next-Day Returns — Baseline Linear AR Model")
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7,7))
sns.scatterplot(x=plot_df["target_ret_next"], y=plot_df["pred_base"], alpha=0.4)
plt.axline((0,0), slope=1, color="red", linestyle="--")
plt.title("Scatter Plot: Actual vs Predicted Returns (Baseline AR)")
plt.xlabel("Actual Next-Day Return")
plt.ylabel("Predicted Return")
plt.tight_layout()
plt.show()


In [ ]:
plot_df["residual"] = plot_df["target_ret_next"] - plot_df["pred_base"]

plt.figure(figsize=(16,5))
sns.lineplot(x="date", y="residual", data=plot_df)
plt.axhline(0, color="red", linestyle="--")
plt.title("Residuals Over Time — Baseline Linear AR")
plt.xlabel("Date")
plt.ylabel("Error (Actual - Predicted)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(plot_df["residual"], kde=True, bins=50)
plt.title("Distribution of Prediction Errors — Baseline Linear AR")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

plot_df["correct_dir"] = (
    np.sign(plot_df["target_ret_next"]) == np.sign(plot_df["pred_base"])
).astype(int)

plot_df["rolling_dir_acc"] = (
    plot_df["correct_dir"].rolling(250).mean()
)

plt.figure(figsize=(16,5))
sns.lineplot(x="date", y="rolling_dir_acc", data=plot_df)
plt.title("Rolling Directional Accuracy (250-Day Window)")
plt.xlabel("Date")
plt.ylabel("Directional Accuracy")
plt.axhline(0.5, color="red", linestyle="--", label="Random Guessing")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
prediction_sample = plot_df[[
    "date", "ticker", "target_ret_next", "pred_base", "residual"
]].head(20)

prediction_sample


## Multimodal Linear AR + Sentiment Model

### Define multimodal feature set including sentiment

In [ ]:
# Numeric mapping of categorical t-1 sentiment
sent_map = {"negative": -1, "neutral": 0, "positive": 1}
train_df["sent_cat_lag1_num"] = train_df["sent_cat_lag1"].map(sent_map)
test_df["sent_cat_lag1_num"] = test_df["sent_cat_lag1"].map(sent_map)

multimodal_features = baseline_features + [
    "sent_lag1",          # numeric t-1 sentiment
    "sent_cat_lag1_num",  # categorical t-1 sentiment mapped
    "sent_roll3"          # past sentiment trend
]

multimodal_features 

### Drop NaNs only inside train/test

In [ ]:
train_mm = train_df.dropna(subset=multimodal_features + ["target_ret_next"]).copy()
test_mm  = test_df.dropna(subset=multimodal_features + ["target_ret_next"]).copy()

X_train_mm = train_mm[multimodal_features]
y_train_mm = train_mm["target_ret_next"]

X_test_mm = test_mm[multimodal_features]
y_test_mm = test_mm["target_ret_next"]


### Scale multimodal features (train-only)

In [ ]:
from sklearn.preprocessing import StandardScaler

scale_cols_mm = multimodal_features  # all numeric

scaler_mm = StandardScaler()
scaler_mm.fit(X_train_mm[scale_cols_mm])

X_train_mm_scaled = X_train_mm.copy()
X_test_mm_scaled = X_test_mm.copy()

X_train_mm_scaled[scale_cols_mm] = scaler_mm.transform(X_train_mm[scale_cols_mm])
X_test_mm_scaled[scale_cols_mm] = scaler_mm.transform(X_test_mm[scale_cols_mm])


### Fit multimodal linear model

In [ ]:
lin_mm = LinearRegression()
lin_mm.fit(X_train_mm_scaled, y_train_mm)

y_pred_mm = lin_mm.predict(X_test_mm_scaled)


### Evaluate multimodal model

In [ ]:
mse_mm  = mean_squared_error(y_test_mm, y_pred_mm)
rmse_mm = np.sqrt(mse_mm)
mae_mm = mean_absolute_error(y_test_mm, y_pred_mm)
r2_mm = r2_score(y_test_mm, y_pred_mm)
dir_acc_mm = directional_accuracy(y_test_mm, y_pred_mm)

print("=== Linear AR + Sentiment (Multimodal) ===")
print(f"MSE : {mse_mm:.6f}")
print(f"RMSE: {rmse_mm:.6f}")
print(f"MAE : {mae_mm:.6f}")
print(f"R²  : {r2_mm:.4f}")
print(f"Directional accuracy: {dir_acc_mm:.4f}")



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plot_mm = test_mm.copy()
plot_mm["pred_mm"] = y_pred_mm
plot_mm = plot_mm.sort_values("date")

sns.set(style="whitegrid", context="talk")

plt.figure(figsize=(16,6))
sns.lineplot(x="date", y="target_ret_next", data=plot_mm, label="Actual Next-Day Return")
sns.lineplot(x="date", y="pred_mm", data=plot_mm, label="Predicted Return (AR + Sentiment)")
plt.title("Actual vs Predicted Next-Day Returns — Linear AR + Sentiment")
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7,7))
sns.scatterplot(x=plot_mm["target_ret_next"], y=plot_mm["pred_mm"], alpha=0.4)
plt.axline((0,0), slope=1, color="red", linestyle="--")
plt.title("Scatter: Actual vs Predicted Returns (AR + Sentiment)")
plt.xlabel("Actual Next-Day Return")
plt.ylabel("Predicted Return")
plt.tight_layout()
plt.show()


In [ ]:
plot_mm["residual_mm"] = plot_mm["target_ret_next"] - plot_mm["pred_mm"]

# Residuals over time
plt.figure(figsize=(16,5))
sns.lineplot(x="date", y="residual_mm", data=plot_mm)
plt.axhline(0, color="red", linestyle="--")
plt.title("Residuals Over Time — Linear AR + Sentiment")
plt.xlabel("Date")
plt.ylabel("Error (Actual - Predicted)")
plt.tight_layout()
plt.show()

# Error distribution
plt.figure(figsize=(10,5))
sns.histplot(plot_mm["residual_mm"], kde=True, bins=50)
plt.title("Distribution of Prediction Errors — Linear AR + Sentiment")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

plot_mm["correct_dir_mm"] = (
    np.sign(plot_mm["target_ret_next"]) == np.sign(plot_mm["pred_mm"])
).astype(int)

plot_mm["rolling_dir_acc_mm"] = plot_mm["correct_dir_mm"].rolling(250).mean()

plt.figure(figsize=(16,5))
sns.lineplot(x="date", y="rolling_dir_acc_mm", data=plot_mm, label="AR + Sentiment")
plt.axhline(0.5, color="red", linestyle="--", label="Random Guessing")
plt.title("Rolling Directional Accuracy (250-Day) — AR + Sentiment")
plt.xlabel("Date")
plt.ylabel("Directional Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
prediction_sample_mm = plot_mm[[
    "date",
    "ticker",
    "target_ret_next",
    "pred_mm",
    "residual_mm"
]].head(20)

prediction_sample_mm


## Analysis

### Per-company model performance

In [ ]:
# Create dataframe with true and predicted values for BOTH models
cmp = test_base[["row_id", "date", "ticker", "target_ret_next"]].copy()
cmp["pred_base"] = y_pred_base

tmp_mm = test_mm[["row_id"]].copy()
tmp_mm["pred_mm"] = y_pred_mm

cmp = cmp.merge(tmp_mm, on="row_id", how="inner")

# Compute direction accuracy per company for ALL companies, and show the full table (not just first 3)
from sklearn.metrics import mean_squared_error

company_metrics = (
    cmp.groupby("ticker")
       .apply(lambda df: pd.Series({
           "RMSE_base":  np.sqrt(mean_squared_error(df["target_ret_next"], df["pred_base"])),
           "RMSE_mm":    np.sqrt(mean_squared_error(df["target_ret_next"], df["pred_mm"])),
           "DirAcc_base": np.mean(np.sign(df["target_ret_next"]) == np.sign(df["pred_base"])),
           "DirAcc_mm":   np.mean(np.sign(df["target_ret_next"]) == np.sign(df["pred_mm"])),
       }))
       .reset_index()
)

# Display the entire company_metrics table sorted by DirAcc_mm descending
display(company_metrics.sort_values("DirAcc_mm", ascending=False).reset_index(drop=True))


### CAP-LEVEL Performance (Small / Mid / Large Cap)

In [ ]:
# Attach cap info to cmp
cmp_cap = cmp.merge(features_model[["row_id", "cap"]], on="row_id", how="left")

cap_metrics = (
    cmp_cap.groupby("cap")
           .apply(lambda df: pd.Series({
               "RMSE_base":  np.sqrt(mean_squared_error(df["target_ret_next"], df["pred_base"])),
               "RMSE_mm":    np.sqrt(mean_squared_error(df["target_ret_next"], df["pred_mm"])),
               "DirAcc_base": np.mean(np.sign(df["target_ret_next"]) == np.sign(df["pred_base"])),
               "DirAcc_mm":   np.mean(np.sign(df["target_ret_next"]) == np.sign(df["pred_mm"])),
           }))
           .reset_index()
)

cap_metrics


### SECTOR-LEVEL Performance

In [ ]:
# Attach sector info to cmp
cmp_sector = cmp.merge(features_model[["row_id", "sector"]], on="row_id", how="left")

sector_metrics = (
    cmp_sector.groupby("sector")
              .apply(lambda df: pd.Series({
                  "RMSE_base":  np.sqrt(mean_squared_error(df["target_ret_next"], df["pred_base"])),
                  "RMSE_mm":    np.sqrt(mean_squared_error(df["target_ret_next"], df["pred_mm"])),
                  "DirAcc_base": np.mean(np.sign(df["target_ret_next"]) == np.sign(df["pred_base"])),
                  "DirAcc_mm":   np.mean(np.sign(df["target_ret_next"]) == np.sign(df["pred_mm"])),
              }))
              .reset_index()
)

sector_metrics.sort_values("DirAcc_mm", ascending=False)
